# import libraries

In [1]:
import ollama
# web_tool.py

import json
import socket
import ipaddress
from urllib.parse import urlparse

import requests
from bs4 import BeautifulSoup
from ddgs import DDGS



# Tools

## Web search

In [2]:

def is_safe_url(url: str) -> bool:
    """Basic protection against private/internal network URLs."""

    try:
        parsed = urlparse(url)

        if parsed.scheme not in ("http", "https"):
            return False

        hostname = parsed.hostname

        if not hostname:
            return False

        if hostname.lower() in ("localhost",):
            return False

        addresses = socket.getaddrinfo(
            hostname,
            None,
            type=socket.SOCK_STREAM
        )

        for address in addresses:
            ip = ipaddress.ip_address(address[4][0])

            if not ip.is_global:
                return False

        return True

    except Exception:
        return False


def read_webpage(url: str, max_chars: int = 6000) -> str:

    if not is_safe_url(url):
        return "Blocked unsafe URL."

    try:
        headers = {
            "User-Agent": (
                "Mozilla/5.0 (X11; Linux x86_64) "
                "AppleWebKit/537.36 Chrome/120 Safari/537.36"
            )
        }

        response = requests.get(
            url,
            headers=headers,
            timeout=10,
            allow_redirects=True
        )

        # Validate the final URL after redirects as well.
        if not is_safe_url(response.url):
            return "Blocked unsafe redirect."

        response.raise_for_status()

        content_type = response.headers.get(
            "Content-Type", ""
        ).lower()

        if "html" not in content_type:
            return "This page is not HTML content."

        soup = BeautifulSoup(response.text, "html.parser")

        # Remove elements that usually contain irrelevant content.
        for tag in soup([
            "script",
            "style",
            "nav",
            "footer",
            "header",
            "aside",
            "noscript",
            "svg",
            "form"
        ]):
            tag.decompose()

        title = (
            soup.title.get_text(" ", strip=True)
            if soup.title else "Untitled"
        )

        text = soup.get_text(
            separator=" ",
            strip=True
        )

        text = " ".join(text.split())

        return (
            f"Page title: {title}\n"
            f"URL: {response.url}\n"
            f"Content:\n{text[:max_chars]}"
        )

    except Exception as e:
        return f"Failed to read webpage: {str(e)}"


def web_search_and_read(query: str,num_of_search_pages: int =3) -> str:
    """
    Search the web for the user query, read relevant pages,
    and return the extracted content.
    """

    try:
        search_results = DDGS().text(
            query,
            max_results=num_of_search_pages
        )

        search_results = list(search_results)

    except Exception as e:
        return f"Search failed: {str(e)}"

    if not search_results:
        return "No search results found."

    output = []

    # Read the first 3 search results.
    for result in search_results[:num_of_search_pages]:

        title = result.get("title", "Untitled")
        url = result.get("href", "")
        snippet = result.get("body", "")

        if not url:
            continue

        page_content = read_webpage(url)

        output.append({
            "title": title,
            "url": url,
            "search_snippet": snippet,
            "page_content": page_content
        })

    return json.dumps(
        output,
        indent=2,
        ensure_ascii=False
    )

In [3]:
search_query = "Python programming language"
search_results = web_search_and_read(search_query)

In [4]:
print(search_results)

[
  {
    "title": "Python (programming language)",
    "url": "https://en.wikipedia.org/wiki/Python_(programming_language)",
    "search_snippet": "Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation, an extensive (\"batteries-included\") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.Guido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revision and not completely backward-compatible with earlier versions. Beginning with Python 3.5, capabilities and keywords for typing were added to the language, allowing optional static typing. As of 2026, the Python Software Foundation supports Python 3.15.2, 3.14.10, 3.13.18, 3.12.16 and 3.11.18 that all were \"expedited security 

In [5]:
search_query = "how to build MCP servers"

search_results = web_search_and_read(query=search_query,num_of_search_pages=5)

In [6]:
print(search_results)

[
  {
    "title": "Build an MCP server - Model Context Protocol",
    "url": "https://modelcontextprotocol.io/docs/2026-07-28/develop/build-server",
    "search_snippet": "Jul 28, 2026 · Get started building your own server to use in Claude for Desktop and other clients.",
    "page_content": "Page title: Build an MCP server - Model Context Protocol\nURL: https://modelcontextprotocol.io/docs/2026-07-28/develop/build-server\nContent:\nBuild an MCP server - Model Context Protocol Documentation Index Fetch the complete documentation index at: /llms.txt Use this file to discover all available pages before exploring further. Skip to main content Assistant Responses are generated using AI and may contain mistakes. In this tutorial, we’ll build a simple MCP weather server and connect it to a host, Claude for Desktop. What we’ll be building We’ll build a server that exposes two tools: get_alerts and get_forecast . Then we’ll connect the server to an MCP host (in this case, Claude for Desktop)

In [7]:
search_query = "what is an mcp server, how it works and how to build MCP servers"

search_results = web_search_and_read(query=search_query,num_of_search_pages=10)

In [8]:
print(search_results)

Search failed: No results found.


## Define and initailize tools

In [9]:
tools = [{
    "type":"function",
    "function":{
        "name":"web_search_and_read",
        "description": (
            "Search the internet for information requested by the user. "
            "Search relevant websites, read their pages, and return "
            "the extracted information and URLs. Use this tool when "
            "the answer requires web research, current information, "
            "documentation, news, or information from websites."
            "ALWAYS use this for current, recent, or latest information, or anything after 2024."
        ),
        "parameters":{
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": (
                            "A specific web search query based on "
                            "the user's information requirement."
                        )
                    }
                },
            "required": ["query"]
        }
    }
}]

tool_functions = {
    "web_search_and_read": web_search_and_read
}

# ollama chat

In [10]:
# def chat(
#     user_input: str,
#     model: str = "llama2",
#     temperature: float = 0.7,
#     max_tokens: int = 500,
#     tools: list = tools,
#     tool_functions: dict = tool_functions
# ) -> str:
#     """
#     Chat with the user, using the specified model and tools.
#     """
#     system_prompt = (
#         "You are a helpful assistant. You can answer questions, "
#         "provide explanations, and perform tasks using available tools. "
#         "You have no knowledge after 2024. For any current or recent information you MUST call web_search_and_read."
#     )
#     try:
#         response = ollama.chat(
#             model=model,
#             messages=[{"role": "user", "content": user_input}]+[{"role": "system", "content": system_prompt}],
#             options={
#                 "temperature": temperature,
#                 'num_predict': 10806,    # Max tokens for the model's generated response
#                 'num_ctx': 6806       # Size of the total memory/prompt context window
#             },
#             tools=tools
#         )
        
#         if response.message.tool_calls:
#             tool_name = response.message.tool_calls[0].name
#             tool_args = response.message.tool_calls[0].arguments
#             print(f"Tool call detected: {tool_name} with arguments: {tool_args}")
#             if tool_name in tool_functions:
#                 tool_function = tool_functions[tool_name]
#                 tool_result = tool_function(**tool_args)

#                 # Send the tool result back to the model for further processing.
#                 response = ollama.chat(
#                     model=model,
#                     messages=[
#                         {"role": "user", "content": user_input},
#                         {"role": "assistant", "content": f"Tool {tool_name} returned: {tool_result}"}
#                     ],
#                     temperature=temperature,
#                     max_tokens=max_tokens,
#                     tools=tools,
#                     tool_functions=tool_functions
#                 )

#         return response.message.content

#     except Exception as e:
#         return f"Chat failed: {str(e)}"
SYSTEM_PROMPT = (
        "You are a helpful assistant. You can answer questions, "
        "provide explanations, and perform tasks using available tools. "
        "You have no knowledge after 2024. For any current or recent information you MUST call web_search_and_read."
    )
# options={
#                 "temperature": temperature,
#                 'num_predict': 10806,    # Max tokens for the model's generated response
#                 'num_ctx': 6806       # Size of the total memory/prompt context window
#             },
def chat(user_input, model="qwen3:8b", temperature=0.6):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},   # first, with today's date
        {"role": "user", "content": user_input},
    ]
    options = {"temperature": temperature, "num_ctx": 16384}

    response = ollama.chat(model=model, messages=messages, options=options,
                           tools=tools, think=True)

    while response.message.tool_calls:
        messages.append(response.message)
        for call in response.message.tool_calls:
            fn = tool_functions.get(call.function.name)
            result = fn(**call.function.arguments) if fn else "Unknown tool"
            messages.append({"role": "tool", "tool_name": call.function.name,
                             "content": str(result)})
        response = ollama.chat(model=model, messages=messages, options=options,
                               tools=tools, think=True)

    return response.message.content

In [11]:
chat_input = "What is an MCP server in AI agents, how does it work, and how can I build one?"
chat_response = chat(chat_input,model="qwen3:8b")
print(chat_response)

Based on the search results, **MCP** in the context of AI agents likely refers to **Model Context Protocol** (or similar acronyms), a framework for structuring AI agent interactions with tools, data, and workflows. Here's a breakdown:

---

### **What is an MCP Server in AI Agents?**
An **MCP Server** is a component that enables **AI agents** (e.g., LLM-driven systems) to interact with external tools, APIs, and data sources in a standardized way. It acts as a bridge between the agent and the backend systems, handling tasks like:
- **Tool discovery** (e.g., OpenAPI-style interfaces).
- **Context management** (e.g., retrieving data from databases like PostgreSQL with pgvector).
- **Security and permissions** (e.g., row-level access control).

Key benefits include:
- **Interoperability**: Connect agents to any MCP-compliant tools.
- **Reusability**: Build once, use across projects and clouds.
- **Operational simplicity**: Reduces boilerplate code for integration.

---

### **How Does an M

In [12]:
response = ollama.chat(
    model="qwen3:8b",
    messages=[{"role": "user", "content": chat_input}],
    options={
        "temperature": 0.0,
        'num_predict': 8000,    # Max tokens for the model's generated response
        'num_ctx': 6806       # Size of the total memory/prompt context window
    },
    tools=tools
)

In [13]:
print(response.message.content)

In [14]:
response.message.tool_calls

[ToolCall(function=Function(name='web_search_and_read', arguments={'query': 'MCP server AI agents definition and implementation'}))]

In [16]:
chat_input = "what is the current (current date: 20-11-2026)AI industry trend, and what are the latest developments in AI research?"
chat_response = chat(chat_input,model="qwen3:8b")
print(chat_response)

As of November 20, 2026, the AI industry and research landscape are characterized by rapid advancements, global competition, and evolving challenges. Here’s a structured overview:

---

### **Key Industry Trends (2026)**
1. **Accelerated AI Capabilities**  
   - Industry-led development dominates, with **over 90% of notable frontier models** (e.g., OpenAI, Anthropic, Google) released in 2025. These models now match or exceed human performance in specialized tasks like PhD-level science questions, multimodal reasoning, and competition mathematics.  
   - **Gemini Deep Think** (Google) achieved gold-medal performance in the International Mathematical Olympiad, though it struggles with basic tasks like analog clock reading (50.1% accuracy).  

2. **U.S.-China Competitiveness**  
   - The **U.S.-China model performance gap has effectively closed**. While the U.S. leads in higher-impact patents and top-tier models, China leads in publication volume, citations, and industrial applications (e

In [ ]:
messages = [
        {"role": "system", "content": SYSTEM_PROMPT},   # first, with today's date
        {"role": "user", "content": chat_input},
    ]
response = ollama.chat(
    model="qwen3:8b",
    messages=messages,
    think=True,
    options={
        "temperature": 0.6,
        'num_predict': 8000,    # Max tokens for the model's generated response
        'num_ctx': 6806       # Size of the total memory/prompt context window
    },
    tools=tools
)

In [ ]:
print(response.message.content)

In [ ]:
print(response.message.tool_calls)

[ToolCall(function=Function(name='web_search_and_read', arguments={'query': 'AI industry trends 2026 latest developments in AI research 2026'}))]


In [ ]:
r = ollama.chat(model="qwen3:8b",
                messages=[{"role": "system", "content": SYSTEM_PROMPT},
                          {"role": "user", "content": chat_input}],
                tools=tools, think=True, options={"temperature": 0.6, "num_ctx": 16384})
print("TOOL CALLS:", r.message.tool_calls)
print("THINKING:", r.message.thinking)
print("CONTENT:", r.message.content)

TOOL CALLS: [ToolCall(function=Function(name='web_search_and_read', arguments={'query': 'AI industry trends 2026'})), ToolCall(function=Function(name='web_search_and_read', arguments={'query': 'latest AI research developments 2025-2026'}))]
THINKING: Okay, the user is asking about the current AI industry trends as of November 20, 2026, and the latest developments in AI research. Since my knowledge cutoff is 2024, I need to use the web_search_and_read tool to get the most up-to-date information. Let me start by formulating the search queries.

First, for the industry trends, I should look for reports or articles from 2025 and 2026. Keywords like "AI industry trends 2026" or "current AI trends November 2026" might work. I'll need to check sources like TechCrunch, Forbes, or industry reports from Gartner and McKinsey. These sources often publish annual reports that include trend analyses.

Next, the latest developments in AI research. I should search for recent breakthroughs, maybe in are